In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
pdf_path = "/content/drive/MyDrive/RAG_project/booka15.pdf"

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fas poppler-utils
!pip -q install pytesseract pdf2image pillow

In [ ]:
import pytesseract

print(pytesseract.get_tesseract_version())

print(pytesseract.get_languages())

In [ ]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
import os
import json
import time

In [ ]:
def ocr_page(pdf_path, page_number, dpi=200):
    images = convert_from_path(
        pdf_path,
        first_page=page_number,
        last_page=page_number,
        dpi=dpi
    )

    image = images[0]

    text = pytesseract.image_to_string(
        image,
        lang="fas",
        config="--psm 6"
    )

    return text

In [ ]:
text = ocr_page(pdf_path, 10)

print(text[:2000])

In [ ]:
ocr_dir = "/content/drive/MyDrive/RAG_project/data/ocr"

os.makedirs(ocr_dir, exist_ok=True)

print(ocr_dir)

In [ ]:
def save_ocr_text(text, page_number, output_dir):
    file_path = os.path.join(
        output_dir,
        f"page_{page_number:03d}.txt"
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    return file_path

In [ ]:
def process_page(pdf_path, page_number, output_dir, dpi=200):
    text = ocr_page(
        pdf_path,
        page_number,
        dpi=dpi
    )

    file_path = save_ocr_text(
        text,
        page_number,
        output_dir
    )

    return text, file_path

In [ ]:
text, file_path = process_page(
    pdf_path,
    5,
    ocr_dir
)

print("Saved to:", file_path)
print(text[:1000])

In [ ]:
total_pages = 222

start_time = time.time()

for page_number in range(1, total_pages + 1):

    output_file = os.path.join(
        ocr_dir,
        f"page_{page_number:03d}.txt"
    )


    if os.path.exists(output_file):
        print(f"Page {page_number}: already exists, skipping")
        continue

    print(f"OCR page {page_number}/{total_pages}...")

    try:
        text = ocr_page(
            pdf_path,
            page_number,
            dpi=200
        )

        save_ocr_text(
            text,
            page_number,
            ocr_dir
        )

        print(f"Page {page_number}: saved")

    except Exception as e:
        print(f"Page {page_number}: ERROR -> {e}")

elapsed = time.time() - start_time

print(f"\nFinished in {elapsed / 60:.2f} minutes")

In [ ]:
ocr_files = [
    file for file in os.listdir(ocr_dir)
    if file.endswith(".txt")
]

print("OCR files:", len(ocr_files))

In [ ]:
page_stats = []

for page_number in range(1, total_pages + 1):

    file_path = os.path.join(
        ocr_dir,
        f"page_{page_number:03d}.txt"
    )

    if not os.path.exists(file_path):
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    page_stats.append({
        "page": page_number,
        "characters": len(text),
        "words": len(text.split())
    })

In [ ]:
import pandas as pd

stats_df = pd.DataFrame(page_stats)

stats_df.head()

In [ ]:
stats_df[
    stats_df["characters"] < 100
]

In [ ]:
stats_path = os.path.join(
    ocr_dir,
    "ocr_stats.csv"
)

stats_df.to_csv(
    stats_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", stats_path)

In [ ]:
!pip -q install langchain langchain-community
from langchain_core.documents import Document

In [ ]:
documents = []

for page_number in range(1, total_pages + 1):

    file_path = os.path.join(
        ocr_dir,
        f"page_{page_number:03d}.txt"
    )

    if not os.path.exists(file_path):
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    document = Document(
        page_content=text,
        metadata={
            "source": pdf_path,
            "page": page_number,
            "ocr": True
        }
    )

    documents.append(document)

print("Documents:", len(documents))

In [ ]:
documents_data = []

for doc in documents:
    documents_data.append({
        "page_content": doc.page_content,
        "metadata": doc.metadata
    })

documents_path = (
    "/content/drive/MyDrive/"
    "RAG_project/data/"
    "ocr_documents.json"
)

with open(documents_path, "w", encoding="utf-8") as f:
    json.dump(
        documents_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", documents_path)


In [ ]:
!pip -q install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        "。",
        "؟",
        "،",
        " ",
        ""
    ]
)

In [ ]:
chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

In [ ]:
!pip -q install -U "sentence-transformers>=5.4.0" transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "Qwen/Qwen3-VL-Embedding-2B"

model = SentenceTransformer(
    model_name,
    device="cuda",
    trust_remote_code=True
)

print("Model loaded successfully")

In [ ]:
texts = [
    chunk.page_content
    for chunk in chunks
]

print("Total chunks:", len(texts))

In [ ]:
embeddings = model.encode(
    texts,
    batch_size=5,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

In [ ]:
import numpy as np
import os

embedding_dir = (
    "/content/drive/MyDrive/"
    "RAG_project/data/embeddings"
)

os.makedirs(embedding_dir, exist_ok=True)

embedding_path = os.path.join(
    embedding_dir,
    "qwen3_vl_embedding_2b.npy"
)

#embeddings = np.load(embedding_path)

np.save(
    embedding_path,
    embeddings
)

print("Saved:", embedding_path)


In [ ]:
import pickle

chunks_path = os.path.join(
    embedding_dir,
    "chunks.pkl"
)

with open(chunks_path, "wb") as f:
    pickle.dump(chunks, f)

print("Saved:", chunks_path)

In [ ]:
print("Chunks:", len(chunks))
print("Embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

In [ ]:
!pip -q install qdrant-client

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

import numpy as np

In [ ]:
qdrant = QdrantClient(
    path="/content/drive/MyDrive/RAG_project/qdrant"
)

print("Qdrant initialized")

In [ ]:
collection_name = "driving_handbook"

qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=2048,
        distance=Distance.COSINE
    )
)

print("Collection created:", collection_name)

In [ ]:
batch_size = 64

for start in range(0, len(chunks), batch_size):

    end = min(
        start + batch_size,
        len(chunks)
    )

    points = []

    for i in range(start, end):

        points.append(
            PointStruct(
                id=i,
                vector=embeddings[i].tolist(),
                payload={
                    "text": chunks[i].page_content,
                    "page": chunks[i].metadata["page"],
                    "source": chunks[i].metadata["source"]
                }
            )
        )

    qdrant.upsert(
        collection_name=collection_name,
        points=points
    )

    print(f"Inserted {end}/{len(chunks)}")

In [ ]:
count = qdrant.count(
    collection_name=collection_name,
    exact=True
)

print("Points in Qdrant:", count.count)
print("Chunks:", len(chunks))

In [ ]:
def retrieve(query, top_k=5):

    # 1. Convert query to embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # 2. Search Qdrant
    search_result = qdrant.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        limit=top_k,
        score_threshold= 0.5
    )

    # 3. Return results
    return search_result.points

def show_results(query, top_k=5):

    results = retrieve(
        query,
        top_k=top_k
    )

    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    for i, result in enumerate(results, 1):

        print(
            f"\n[{i}] "
            f"Score: {result.score:.4f} | "
            f"Page: {result.payload['page']}"
        )

        print("-" * 100)
        print(result.payload["text"][:800])

In [ ]:
test_queries = [
    "آزادراه چیست؟",
    "راننده چه کسی است؟",
    "شانه راه چیست؟",
    "خیابان اصلی چیست؟",
    "تابلوهای اخطاری چه هستند؟",
    "در هنگام خرابی وسیله نقلیه در جاده چه کاری باید انجام داد؟"
]
for query in test_queries:

    show_results(
        query,
        top_k=3
    )

##Reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cuda"
)

print("Reranker loaded!")

In [ ]:
def retrieve_candidates(query, top_k=20):

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    result = qdrant.query_points(
        collection_name="driving_handbook",
        query=query_embedding.tolist(),
        limit=top_k
    )

    return result.points

In [ ]:
candidates = retrieve_candidates(
    "شانه راه چیست؟",
    top_k=20
)

print("Candidates:", len(candidates))

In [ ]:
query = "شانه راه چیست؟"

pairs = [
    (
        query,
        result.payload["text"]
    )
    for result in candidates
]

print("Pairs:", len(pairs))

In [ ]:
rerank_scores = reranker.predict(
    pairs,
    batch_size=8,
    show_progress_bar=True
)

print(rerank_scores)

In [ ]:
reranked = sorted(
    zip(candidates, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (result, score) in enumerate(reranked[:5], 1):

    print("=" * 80)
    print(f"RERANKED RESULT {i}")
    print("Reranker score:", round(float(score), 4))
    print("Qdrant score:", round(float(result.score), 4))
    print("Page:", result.payload["page"])
    print("=" * 80)

    print(result.payload["text"][:1000])

##Turn everything into one function

In [ ]:
def retrieve_and_rerank(
    query,
    retrieval_k=20,
    final_k=5
):

    # Step 1: Qdrant retrieval
    candidates = retrieve_candidates(
        query,
        top_k=retrieval_k
    )

    # Step 2: Create query-document pairs
    pairs = [
        (
            query,
            result.payload["text"]
        )
        for result in candidates
    ]

    # Step 3: Rerank
    scores = reranker.predict(
        pairs,
        batch_size=8,
        show_progress_bar=False
    )

    # Step 4: Combine results + scores
    reranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Step 5: Return best results
    return reranked[:final_k]

In [ ]:
results = retrieve_and_rerank(
    "در هنگام خرابی وسیله نقلیه در جاده چه کاری باید انجام داد؟",
    retrieval_k=20,
    final_k=5
)

for i, (result, score) in enumerate(results, 1):

    print("=" * 80)
    print(f"RESULT {i}")
    print("Reranker score:", round(float(score), 4))
    print("Page:", result.payload["page"])
    print("=" * 80)

    print(result.payload["text"][:1200])

#Build the Context

In [ ]:
def build_context(results):

    context_parts = []

    for i, (result, score) in enumerate(results, 1):

        text = result.payload["text"]
        page = result.payload["page"]

        context_parts.append(
            f"[Source {i} | Page {page}]\n"
            f"{text}"
        )

    return "\n\n".join(context_parts)

In [ ]:
query = "در هنگام خرابی وسیله نقلیه در جاده چه کاری باید انجام داد؟"

results = retrieve_and_rerank(
    query,
    retrieval_k=20,
    final_k=1
)

context = build_context(results)

print(context)

In [ ]:
def build_prompt(query, context):

    prompt = f"""
تو یک دستیار هوشمند برای کتاب آموزش قوانین رانندگی هستی.

به سؤال کاربر فقط بر اساس اطلاعات موجود در Context پاسخ بده.

اگر پاسخ سؤال در Context وجود ندارد، صادقانه بگو:
«اطلاعات کافی برای پاسخ به این سؤال در کتاب پیدا نشد.»

اطلاعاتی که در Context وجود ندارد را حدس نزن.

برای هر جوابی که میدهی باید منبع آن را هم بنویسی طبق context
Context:
----------------
{context}
----------------

Question:
{query}

Answer:
"""

    return prompt

In [ ]:
prompt = build_prompt(
    query,
    context
)

print(prompt)

In [ ]:
!pip -q install openai

In [ ]:
from google.colab import userdata
api = userdata.get('OPENROUTER_API_KEY')

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api
)

In [ ]:
def generate_answer_stream(prompt):

    stream = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        stream=True
    )

    for chunk in stream:

        content = chunk.choices[0].delta.content

        if content:
            yield content

In [ ]:
def rag_answer(query):

    # Retrieve + rerank
    results = retrieve_and_rerank(
        query,
        retrieval_k=20,
        final_k=5
    )

    # Build context
    context = build_context(results)

    # Build prompt
    prompt = build_prompt(
        query,
        context
    )

    # Generate answer
    answer = generate_answer_stream(prompt)

    return answer, results

In [ ]:
answer, results = rag_answer(
    "مواجه شدن با صحنه تصادف"
)

print(answer)

In [ ]:
!pip -q install gradio

In [ ]:
def chat(query):

    if not query.strip():
        yield ""
        return

    # 1. Retrieve + rerank
    results = retrieve_and_rerank(
        query,
        retrieval_k=20,
        final_k=5
    )

    # 2. Build context
    context = build_context(results)

    # 3. Build prompt
    prompt = build_prompt(
        query,
        context
    )

    # 4. Stream answer
    answer = ""

    for token in generate_answer_stream(prompt):

        answer += token

        yield answer

    # 5. Add sources after generation
    pages = []

    for result, score in results:

        page = result.payload["page"]

        if page not in pages:
            pages.append(page)

    sources = "\n\n**Sources:** " + ", ".join(
        f"Page {page}" for page in pages
    )

    yield answer + sources

In [ ]:
import gradio as gr

with gr.Blocks(
    title="Driving Handbook RAG Assistant"
) as demo:

    gr.Markdown(
        """
        # 🚗 Driving Handbook RAG Assistant

        Ask a question about the driving handbook.
        """
    )

    question = gr.Textbox(
        label="Question",
        placeholder="سؤال خود را درباره قوانین رانندگی بپرسید..."
    )

    answer = gr.Markdown(
        label="Answer"
    )

    ask_button = gr.Button(
        "Ask",
        variant="primary"
    )

    clear_button = gr.Button(
        "Clear"
    )

    ask_button.click(
        fn=chat,
        inputs=question,
        outputs=answer
    )

    question.submit(
        fn=chat,
        inputs=question,
        outputs=answer
    )

    clear_button.click(
        fn=lambda: ("", ""),
        inputs=None,
        outputs=[question, answer]
    )

demo.launch(
    share=True,
    debug=True
)